# GRU Model Results vs Baseline

Evaluates the trained GRU on validation and test sets.
Compares against logistic regression baseline on all metrics.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import joblib
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score,
    average_precision_score, brier_score_loss
)
from sklearn.calibration import calibration_curve

ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.features.encode_possessions import N_FEATURES, load_split_tensors, MAX_SEQ_LEN
from src.models.lstm_model import PossessionGRU
from src.models.dataset import PossessionDataset
from torch.utils.data import DataLoader

MODELS = ROOT / 'models' / 'trained'
PROCESSED = ROOT / 'data' / 'processed'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

NUMERIC_FEATURES = [
    'n_events','n_pass','n_carry','n_dribble','n_pressure',
    'n_attacking_third','start_x','end_x','progression',
    'total_duration','match_minute_start',
]
CATEGORICAL_FEATURES = ['start_zone','end_zone','play_pattern']

In [ ]:
# Load training log
with open(MODELS / 'gru_train_log.json') as f:
    log = json.load(f)

epochs = [e['epoch'] for e in log]
train_loss = [e['train_loss'] for e in log]
val_loss = [e['val_loss'] for e in log]
val_roc = [e['val_roc_auc'] for e in log]
val_pr = [e['val_pr_auc'] for e in log]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, train_loss, label='Train'); axes[0].plot(epochs, val_loss, label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs, val_roc, color='#3b82f6'); axes[1].set_title('Val ROC-AUC'); axes[1].set_xlabel('Epoch')
axes[2].plot(epochs, val_pr, color='#ef4444'); axes[2].set_title('Val PR-AUC'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(MODELS / 'gru_training_curves.png', dpi=120)
plt.show()

best_epoch = max(log, key=lambda e: e['val_pr_auc'])
print(f"Best epoch: {best_epoch['epoch']} — Val PR-AUC={best_epoch['val_pr_auc']:.4f} ROC={best_epoch['val_roc_auc']:.4f}")

In [ ]:
def gru_probs(split):
    X, L, y, df = load_split_tensors(split, MAX_SEQ_LEN)
    model = PossessionGRU(input_size=N_FEATURES, hidden_size=128, num_layers=2, dropout=0.0)
    model.load_state_dict(torch.load(MODELS / 'gru_best.pt', map_location=device))
    model.to(device).eval()
    loader = DataLoader(PossessionDataset(X, L, y), batch_size=512, shuffle=False)
    all_p = []
    with torch.no_grad():
        for xb, lb, _ in loader:
            all_p.append(torch.sigmoid(model(xb.to(device), lb.to(device))).cpu().numpy())
    return np.concatenate(all_p), y, df

def bl_probs(split):
    bl = joblib.load(MODELS / 'baseline_logreg.pkl')
    match_ids = pd.read_csv(PROCESSED/'splits'/f'{split}_matches.csv')['match_id'].tolist()
    df = pd.read_parquet(PROCESSED/'possessions'/'possessions.parquet')
    df = df[df['match_id'].isin(match_ids)].reset_index(drop=True)
    return bl.predict_proba(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])[:,1], df['ends_in_shot'].values.astype(np.float32), df

gru_p_val, y_val, gru_val_df = gru_probs('validation')
gru_p_test, y_test, gru_test_df = gru_probs('test')
bl_p_val, _, bl_val_df = bl_probs('validation')
bl_p_test, _, bl_test_df = bl_probs('test')
print('Predictions computed')

## ROC and PR Curves — GRU vs Baseline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, gp, bp, y in [('Validation', gru_p_val, bl_p_val, y_val), ('Test', gru_p_test, bl_p_test, y_test)]:
    style = '-' if name == 'Validation' else '--'
    fpr_g, tpr_g, _ = roc_curve(y, gp)
    fpr_b, tpr_b, _ = roc_curve(y, bp)
    axes[0].plot(fpr_g, tpr_g, style+'b', label=f'GRU {name} ({roc_auc_score(y,gp):.3f})')
    axes[0].plot(fpr_b, tpr_b, style+'r', label=f'Baseline {name} ({roc_auc_score(y,bp):.3f})')

    pr_g, rc_g, _ = precision_recall_curve(y, gp)
    pr_b, rc_b, _ = precision_recall_curve(y, bp)
    axes[1].plot(rc_g, pr_g, style+'b', label=f'GRU {name} ({average_precision_score(y,gp):.3f})')
    axes[1].plot(rc_b, pr_b, style+'r', label=f'Baseline {name} ({average_precision_score(y,bp):.3f})')

axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
axes[0].set_title('ROC Curve'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend(fontsize=8)
axes[1].set_title('Precision-Recall Curve'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(MODELS / 'gru_vs_baseline_curves.png', dpi=120)
plt.show()

## Summary Metrics Table

In [ ]:
rows = []
for split, gp, bp, y in [('val', gru_p_val, bl_p_val, y_val), ('test', gru_p_test, bl_p_test, y_test)]:
    for name, p in [('GRU', gp), ('Baseline', bp)]:
        rows.append({'Model': name, 'Split': split,
                     'ROC-AUC': roc_auc_score(y, p),
                     'PR-AUC': average_precision_score(y, p),
                     'Brier': brier_score_loss(y, p)})

summary = pd.DataFrame(rows).set_index(['Model','Split']).round(4)
print(summary.to_string())

## Per-Competition Results (Validation)

In [ ]:
comp_rows = []
for comp, idx in gru_val_df.groupby('competition_label').groups.items():
    idx = list(idx)
    yc = y_val[idx]
    gp_c = gru_p_val[idx]
    bp_c = bl_p_val[list(bl_val_df.groupby('competition_label').groups[comp])]
    comp_rows.append({'Competition': comp, 'n': len(yc),
                      'GRU ROC': roc_auc_score(yc, gp_c), 'GRU PR': average_precision_score(yc, gp_c),
                      'BL ROC': roc_auc_score(yc, bp_c), 'BL PR': average_precision_score(yc, bp_c)})

comp_df = pd.DataFrame(comp_rows).round(4)
print(comp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9,4))
x = np.arange(len(comp_df))
w = 0.35
ax.bar(x - w/2, comp_df['GRU PR'], w, label='GRU', color='#3b82f6')
ax.bar(x + w/2, comp_df['BL PR'], w, label='Baseline', color='#ef4444')
ax.set_xticks(x); ax.set_xticklabels(comp_df['Competition'], rotation=15)
ax.set_ylabel('PR-AUC'); ax.set_title('PR-AUC per Competition — GRU vs Baseline (Validation)')
ax.legend()
plt.tight_layout()
plt.savefig(MODELS / 'gru_per_competition.png', dpi=120)
plt.show()